# Inhibitory Two-Edge Triad Analysis

This notebook mirrors the style of `inhibitory_modulation_analysis.ipynb`, but shifts the unit of analysis from the whole E/I block system to induced three-node, two-edge directed triads. The input CSV is configured as rows=senders/presynaptic and columns=receivers/postsynaptic (`MATRIX_ORIENTATION = "pre_by_post"`). The loader transposes that representation internally so helper functions operate with rows=receivers/postsynaptic and columns=senders/presynaptic.

## tl;dr

With the default connected-only, self-free setting, the matrix contains 5,743 induced three-node, two-edge triads: 2,811 divergent `021D` motifs, which map to Rees superpattern `C`; 1,644 chain `021C` motifs, which map to Rees superpattern `B`; and 1,288 convergent `021U` motifs, which map to Rees superpattern `A`. The strongest composition by count and weight is `1E-2I`, while the largest edge-signature mass comes from `E->I + E->I` triads. Density-matched nulls with 20 draws do not mark motif or node-E/I categories as enriched; they are all less frequent than the random-edge baseline. Ablating all triad-participating non-self edges removes 711 unique directed edges and lowers the full-system spectral radius by 193.8. The triad-aggregate Schur reduction is highly sensitive to inhibitory-block regularization, so its entries should be interpreted as a diagnostic ranking rather than as a single fixed reduced circuit.


## Setup

Import the analysis functions and set the input file. The default enumerates connected induced two-edge triads: 021D, 021U, and 021C in triad-census notation. Set `CONNECTED_ONLY = False` if reciprocal dyads with an isolated third node (`102`) should also be included. In this notebook `INCLUDE_SELF_CONNECTIONS = False`, so self-loops are not included in the triad edge set.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from inhibitory_modulation import (
    REES_SUPERPATTERN_NAMES,
    TRIAD_CENSUS_TO_REES_SUPERPATTERN,
    enumerate_two_edge_triads,
    load_connectivity,
    make_ei_blocks,
    node_metadata,
    summarize_blocks,
    summarize_triad_categories,
    triad_ablation_analysis,
    triad_ablation_significance,
    triad_edge_participation,
    triad_enrichment_significance,
    triad_schur_decomposition,
)

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_matrix.csv"
# CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_netlist.csv"
MATRIX_ORIENTATION = "pre_by_post"  # input CSV rows=senders/presynaptic, columns=receivers/postsynaptic
CLASS_MAP = {}

CONNECTED_ONLY = True
INCLUDE_SELF_CONNECTIONS = False
N_NULL = 20
RANDOM_STATE = 0
SCHUR_REGULARIZATION = 1e-6

CONNECTIVITY_PATH


## Load Connectivity

This step reads the selected connectivity export into a labeled population matrix and attaches coarse metadata: E/I class and a region inferred from the first token of the population name. The source file is rows=senders and columns=receivers; the loaded matrix shown below is normalized internally to rows=receivers and columns=senders.


In [ ]:
data = load_connectivity(
    CONNECTIVITY_PATH,
    class_map=CLASS_MAP,
    matrix_orientation=MATRIX_ORIENTATION,
)
metadata = node_metadata(data.classes)
blocks = make_ei_blocks(data.matrix, data.classes)

print(f"Loaded: {data.source}")
print(f"Matrix shape: {data.matrix.shape[0]} x {data.matrix.shape[1]}")
display(metadata["class_label"].value_counts().to_frame("population_count"))
display(metadata["region"].value_counts().to_frame("population_count").head(12))
display(summarize_blocks(blocks))


## Enumerate Two-Edge Triads

The enumerator scans each three-population induced subgraph, ignores self-loops, and keeps triads with exactly two directed off-diagonal edges. With `CONNECTED_ONLY = True`, the retained motifs are the connected two-edge classes: convergent (`021U`, Rees `A`), directed-chain (`021C`, Rees `B`), and divergent/single-input (`021D`, Rees `C`).


In [ ]:
triads = enumerate_two_edge_triads(
    data.matrix,
    data.classes,
    connected_only=CONNECTED_ONLY,
    include_self_connections=INCLUDE_SELF_CONNECTIONS,
)

rees_crosswalk = pd.DataFrame(
    [
        {
            "motif": motif,
            "rees_superpattern": letter,
            "rees_superpattern_name": REES_SUPERPATTERN_NAMES[letter],
        }
        for motif, letter in TRIAD_CENSUS_TO_REES_SUPERPATTERN.items()
        if motif in triads["motif"].unique()
    ]
)

print(f"Enumerated connected two-edge triads: {len(triads):,}")
print(f"Include self connections: {INCLUDE_SELF_CONNECTIONS}")
display(rees_crosswalk.sort_values("rees_superpattern"))
display(
    triads
    .groupby(["rees_superpattern", "motif", "rees_superpattern_name"])
    .size()
    .to_frame("triad_count")
)
display(triads["edge_count"].value_counts().sort_index().to_frame("triad_count"))
display(triads.head())


## Triad Topology And E/I Structure

These tables rank motifs and E/I signatures by count and total absolute edge weight. Because connected two-edge directed triads are acyclic, their local 3-by-3 spectra are nilpotent and have zero spectral radius; the ranking therefore emphasizes count, weight mass, and composition.


In [ ]:
motif_summary = summarize_triad_categories(
    triads,
    ["rees_superpattern", "motif", "rees_superpattern_name"],
)
node_ei_summary = summarize_triad_categories(triads, "node_ei_signature")
edge_ei_summary = summarize_triad_categories(triads, "edge_ei_signature")

print("Motif census")
display(motif_summary)

print("Node E/I composition")
display(node_ei_summary)

print("Directed edge E/I signatures")
display(edge_ei_summary.head(15))


## Regions Involved

Region signatures collapse each triad to the unique set of regions represented by its three populations. This highlights whether the two-edge motifs are local to one anatomical label or bridge multiple labels.


In [ ]:
region_summary = summarize_triad_categories(triads, "region_signature")

print("Most frequent region signatures")
display(region_summary.head(20))

print("Within-region share by motif")
display(
    triads
    .groupby("motif")
    .agg(
        triad_count=("triad_id", "count"),
        within_region_rate=("within_region", "mean"),
        median_region_count=("region_count", "median"),
    )
    .sort_values("triad_count", ascending=False)
)


## Ranking Triads And Edges

The first table ranks individual triads by total absolute edge weight. The second table ranks directed edges by how often they participate in enumerated two-edge triads.


In [ ]:
ranked_triads = triads.sort_values("total_abs_weight", ascending=False)
edge_participation = triad_edge_participation(data.matrix, triads)

ranking_columns = [
    "triad_id",
    "motif",
    "rees_superpattern",
    "rees_superpattern_name",
    "node_ei_signature",
    "edge_ei_signature",
    "region_signature",
    "node_a",
    "node_b",
    "node_c",
    "total_abs_weight",
]

display(ranked_triads[ranking_columns].head(20))
display(edge_participation.head(20))


## Enrichment Significance

The null model samples the same number of directed non-self edges into random off-diagonal positions, then resamples observed weights conditional on presynaptic E/I class. This tests whether triad categories are overrepresented relative to a density-matched random directed graph with class-consistent weight signs. Increase `N_NULL` for publication-grade p-values.


In [ ]:
enrichment_motif = triad_enrichment_significance(
    data.matrix,
    data.classes,
    by="rees_superpattern",
    n_null=N_NULL,
    random_state=RANDOM_STATE,
    connected_only=CONNECTED_ONLY,
    include_self_connections=INCLUDE_SELF_CONNECTIONS,
).merge(
    rees_crosswalk[["rees_superpattern", "motif", "rees_superpattern_name"]],
    on="rees_superpattern",
    how="left",
)

enrichment_node_ei = triad_enrichment_significance(
    data.matrix,
    data.classes,
    by="node_ei_signature",
    n_null=N_NULL,
    random_state=RANDOM_STATE + 1,
    connected_only=CONNECTED_ONLY,
    include_self_connections=INCLUDE_SELF_CONNECTIONS,
)

print(f"Null draws per enrichment table: {N_NULL}")
display(enrichment_motif)
display(enrichment_node_ei)


## Ablation Significance

Ablation removes directed edges that participate in enumerated triads and recomputes whole-system stability diagnostics. The significance table compares motif-specific edge removals with random removals of the same number of existing nonzero edges.


In [ ]:
global_triad_ablation = triad_ablation_analysis(data.matrix, triads)
top_superpatterns = motif_summary.index.get_level_values("rees_superpattern").tolist()
motif_ablation = triad_ablation_analysis(
    data.matrix,
    triads,
    category_col="rees_superpattern",
    categories=top_superpatterns,
)
motif_ablation_significance = triad_ablation_significance(
    data.matrix,
    triads,
    category_col="rees_superpattern",
    categories=top_superpatterns,
    n_null=N_NULL,
    random_state=RANDOM_STATE + 2,
).merge(
    rees_crosswalk[["rees_superpattern", "motif", "rees_superpattern_name"]].rename(columns={"rees_superpattern": "category"}),
    on="category",
    how="left",
)

print("All triad-participating edges")
display(global_triad_ablation)

print("Rees-superpattern-specific triad edge ablations")
display(motif_ablation)

print(f"Random edge-set null draws per superpattern: {N_NULL}")
display(motif_ablation_significance)


## Schur Reduction Of The Triad Ensemble

The original modulation notebook uses an E/I block Schur-complement reduction, `EE - EI * inv(II + lambda I) * IE`, to collapse inhibitory effects onto the excitatory subspace. Here the same reduction is applied to a triad-aggregate matrix: every enumerated triad edge is embedded back into the full population matrix and averaged by triad count before E/I block partitioning.

This is a Schur-complement reduction of the enumerated triads as a unit, not an orthogonal Schur factorization. For the underlying linear algebra, see Zhang's edited volume on Schur complements and standard numerical linear algebra references for Schur forms and decompositions.


In [ ]:
triad_schur = triad_schur_decomposition(
    data.matrix,
    data.classes,
    triads,
    regularization=SCHUR_REGULARIZATION,
    normalize="triad_count",
)

schur_stability = pd.DataFrame(
    [triad_schur.aggregate_stability, triad_schur.effective_stability]
).set_index("name")

print("Triad-aggregate E/I block summaries")
display(summarize_blocks(triad_schur.blocks))

print("Stability summaries")
display(schur_stability)

print("Largest absolute entries in the inhibitory feedback term")
display(
    triad_schur.feedback
    .stack()
    .rename("feedback_weight")
    .abs()
    .sort_values(ascending=False)
    .head(20)
    .to_frame()
)

print("Largest absolute entries in the Schur-effective E matrix")
display(
    triad_schur.effective_excitation
    .stack()
    .rename("effective_weight")
    .abs()
    .sort_values(ascending=False)
    .head(20)
    .to_frame()
)


## Schur Regularization Sensitivity

The triad aggregate contains a sparse inhibitory block, so the pseudo-inverse term can be sensitive to the diagonal regularization. This sweep keeps the same triad ensemble and varies only `lambda` in `II + lambda I`.


In [ ]:
regularization_grid = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])
sensitivity_rows = []

for regularization in regularization_grid:
    decomp = triad_schur_decomposition(
        data.matrix,
        data.classes,
        triads,
        regularization=float(regularization),
        normalize="triad_count",
    )
    row = decomp.effective_stability.copy()
    row["regularization"] = regularization
    sensitivity_rows.append(row)

schur_sensitivity = pd.DataFrame(sensitivity_rows).set_index("regularization")
display(schur_sensitivity)


## Takeaways

- The connected two-edge triad census is dominated by Rees `C` / `021D` divergent single-input motifs, followed by Rees `B` / `021C` chains and Rees `A` / `021U` convergent simple-regulation motifs.
- E/I structure is not balanced across triads: `1E-2I` triads carry most of the total absolute weight, and `E->I + E->I` is the largest edge-level E/I signature by weight mass.
- Most frequent region signatures bridge regions rather than staying within one region; `CA1-CA3`, `CA1-DG`, and `CA1-CA2-CA3` are the most common signatures in the default run.
- The self-free analysis keeps exactly two non-self directed edges per triad and excludes all diagonal entries from ranking, aggregation, and ablation.
- The density-matched edge null produces many more two-edge connected triads than observed, so the default enrichment readout is depletion rather than enrichment for the tested Rees superpatterns and node-E/I categories.
- Rees-superpattern-specific ablations have modest effects relative to random edge-set ablations of the same size; the observed removals are less stabilizing than the random baseline in the default run.
- The Schur-complement reduction of the triad ensemble is useful for ranking feedback and effective-excitation entries, but the inhibitory pseudo-inverse is regularization-sensitive and should be swept before drawing biological conclusions.


## References

- Holland, P. W., & Leinhardt, S. (1974). *The Statistical Analysis of Local Structure in Social Networks*. NBER Working Paper 0044. https://doi.org/10.3386/w0044
- NetworkX documentation: `triadic_census`, including the 16 directed triad-census labels used here. https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.triads.triadic_census.html
- Rees, C. L., Wheeler, D. W., Hamilton, D. J., White, C. M., Komendantov, A. O., & Ascoli, G. A. (2016). *Graph Theoretic and Motif Analyses of the Hippocampal Neuron Type Potential Connectome*. eNeuro, 3(6), ENEURO.0205-16.2016. https://www.eneuro.org/content/3/6/ENEURO.0205-16.2016
- Zhang, F. (Ed.). (2005). *The Schur Complement and Its Applications*. Springer. https://doi.org/10.1007/b105056
- Golub, G. H., & Van Loan, C. F. (2013). *Matrix Computations* (4th ed.). Johns Hopkins University Press. https://www.press.jhu.edu/books/title/10678/matrix-computations
- SciPy documentation: `scipy.linalg.schur`, for the standard Schur matrix decomposition `A = Z T Z^H`. https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.schur.html
